# FinTok AI · 03_image / 01 · Avatares cartoon

**Objetivo:** convertir una foto de cada una de nosotras en un avatar de dibujo animado, con dos versiones (boca cerrada y boca abierta) y fondo transparente. El montador de vídeo alternará entre ambas según el volumen del audio para que parezca que el avatar habla.

**Modelos:**
- **SDXL Turbo** (image-to-image, como en el notebook 4 de clase): redibuja la foto en estilo cartoon.
- **rembg (U²-Net)**: recorta el fondo para poder poner el avatar encima de cualquier escena.

**Entrada:** una foto frontal por persona (cara bien iluminada, boca cerrada, fondo sencillo).

**Salida** (en `assets/avatar/`):
- `<persona>_closed.png` y `<persona>_open.png` (RGBA, 512×512)
- `<persona>_preview.gif` (comprobación rápida)
- `<persona>_meta.json` (prompt, strength y seed usados, para reproducibilidad)

**Cómo usarlo:** Entorno de ejecución → Cambiar tipo → **GPU T4**. Ejecuta el notebook con `PERSONA = "andrea"`, y luego cambia a la otra persona y vuelve a ejecutar desde la sección 2.

> Las funciones de este notebook pasarán después a `src/image/avatar.py`.

## 0. Instalación

In [ ]:
!pip install -q diffusers transformers accelerate rembg onnxruntime

## 1. Imports, configuración y modelo

In [ ]:
import os, json
import torch
from PIL import Image, ImageOps, ImageDraw, ImageFilter
from IPython.display import display, Image as IPImage
from diffusers import AutoPipelineForImage2Image

AVATAR_DIR = "assets/avatar"
os.makedirs(AVATAR_DIR, exist_ok=True)
SIZE = 512  # SDXL Turbo está entrenado a 512x512

In [ ]:
img2img = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
).to("cuda")

## 2. Elegir persona y subir su foto

In [ ]:
PERSONA = "andrea"        # cambia a "companera" en la segunda pasada
DESCRIPCION = "young woman with long brown hair"   # rasgos para ayudar al modelo a parecerse

In [ ]:
from google.colab import files
uploaded = files.upload()          # sube la foto de PERSONA
FOTO = list(uploaded.keys())[0]
print("Foto:", FOTO)

In [ ]:
def load_portrait(path, size=SIZE):
    """Corrige la rotación del móvil, recorta cuadrado centrado en la cara y redimensiona."""
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    return ImageOps.fit(img, (size, size), centering=(0.5, 0.4))

foto = load_portrait(FOTO)
display(foto)

## 3. Generar el avatar cartoon (boca cerrada)

Probamos varias combinaciones de `strength` (cuánto se aleja de la foto) y `seed` (variación aleatoria):
- `strength` bajo → se parece más a la foto, menos cartoon.
- `strength` alto → más cartoon, menos parecido.

Con `guidance_scale=0.0` y 4 pasos, como indica el notebook de clase.

In [ ]:
STYLE_PROMPT = (
    "3D animated cartoon character portrait, Pixar style, {desc}, financial analyst, "
    "wearing a professional navy blazer, {mouth}, looking at the camera, front view, "
    "head and shoulders, soft studio lighting, plain light grey background"
)
PROMPT_CLOSED = STYLE_PROMPT.format(desc=DESCRIPCION, mouth="friendly smile with closed mouth")

def generate_variants(image, prompt, strengths=(0.4, 0.5, 0.6), seeds=(0, 1, 2), steps=4):
    results = []
    for s in strengths:
        for seed in seeds:
            g = torch.Generator("cuda").manual_seed(seed)
            out = img2img(prompt=prompt, image=image, strength=s, guidance_scale=0.0,
                          num_inference_steps=steps, generator=g).images[0]
            results.append({"strength": s, "seed": seed, "image": out})
    return results

def show_grid(results, cols=3, thumb=256):
    rows = (len(results) + cols - 1) // cols
    grid = Image.new("RGB", (cols * thumb, rows * (thumb + 24)), "white")
    d = ImageDraw.Draw(grid)
    for i, r in enumerate(results):
        x, y = (i % cols) * thumb, (i // cols) * (thumb + 24)
        grid.paste(r["image"].resize((thumb, thumb)), (x, y + 24))
        d.text((x + 5, y + 5), f"#{i}  strength={r['strength']}  seed={r['seed']}", fill="black")
    display(grid)

In [ ]:
variants_closed = generate_variants(foto, PROMPT_CLOSED)
show_grid(variants_closed)

Elige el número (`#`) del que más te guste. Si ninguno convence, cambia `DESCRIPCION`, los `strengths` o las `seeds` y vuelve a ejecutar la celda anterior.

In [ ]:
ELEGIDO_CLOSED = 4
closed = variants_closed[ELEGIDO_CLOSED]["image"]
display(closed)

## 4. Versión con la boca abierta

Truco para que el avatar no "tiemble" al alternar: generamos una versión con la boca abierta partiendo del avatar elegido y **solo copiamos la zona de la boca** sobre la versión cerrada. Así el resto de la cara es idéntico en las dos imágenes.

**4.1** Generamos candidatas con la boca abierta (strength bajo para que no cambie demasiado).

In [ ]:
PROMPT_OPEN = STYLE_PROMPT.format(desc=DESCRIPCION, mouth="mouth open while talking, slightly showing teeth")
variants_open = generate_variants(closed, PROMPT_OPEN, strengths=(0.3, 0.4, 0.5), seeds=(0, 1, 2))
show_grid(variants_open)

In [ ]:
ELEGIDO_OPEN = 3
opened_full = variants_open[ELEGIDO_OPEN]["image"]

**4.2** Localiza la boca. La rejilla marca los píxeles cada 32 (las líneas rojas fuertes, cada 128). Apunta las coordenadas de un rectángulo que cubra la boca con algo de margen.

In [ ]:
def show_with_grid(img, step=32):
    im = img.copy().convert("RGB")
    d = ImageDraw.Draw(im)
    for v in range(0, im.width, step):
        color = (255, 0, 0) if v % 128 == 0 else (255, 170, 170)
        d.line([(v, 0), (v, im.height)], fill=color)
        d.line([(0, v), (im.width, v)], fill=color)
        d.text((v + 2, 2), str(v), fill="red")
        d.text((2, v + 2), str(v), fill="red")
    display(im.resize((768, 768)))

show_with_grid(closed)

In [ ]:
MOUTH_BOX = (200, 320, 312, 380)   # (x0, y0, x1, y1) — ajústalo a tu avatar

def preview_box(img, box):
    im = img.copy().convert("RGB")
    ImageDraw.Draw(im).ellipse(box, outline="lime", width=3)
    display(im)

preview_box(closed, MOUTH_BOX)

**4.3** Pegamos la boca abierta con bordes suaves.

In [ ]:
def paste_mouth(closed, opened, box, feather=6):
    mask = Image.new("L", closed.size, 0)
    ImageDraw.Draw(mask).ellipse(box, fill=255)
    mask = mask.filter(ImageFilter.GaussianBlur(feather))
    return Image.composite(opened, closed, mask)

opened = paste_mouth(closed, opened_full, MOUTH_BOX)
display(opened)

**Plan B** si ninguna boca generada queda bien: dibujamos una boca abierta sencilla encima. Para la duración de un TikTok queda perfectamente.

In [ ]:
USAR_PLAN_B = False

def draw_open_mouth(img, box):
    im = img.copy().convert("RGB")
    x0, y0, x1, y1 = box
    w, h = x1 - x0, y1 - y0
    inner = (x0 + w * 0.2, y0 + h * 0.3, x1 - w * 0.2, y1 - h * 0.15)
    d = ImageDraw.Draw(im)
    d.ellipse(inner, fill=(70, 25, 30))
    ix0, iy0, ix1, iy1 = inner
    d.ellipse((ix0 + (ix1 - ix0) * 0.25, iy0 + (iy1 - iy0) * 0.55, ix1 - (ix1 - ix0) * 0.25, iy1),
              fill=(200, 90, 100))  # lengua
    return im

if USAR_PLAN_B:
    opened = draw_open_mouth(closed, MOUTH_BOX)
    display(opened)

## 5. Quitar el fondo

Calculamos la silueta **una sola vez** sobre la versión cerrada y la aplicamos a las dos, para que el contorno sea idéntico.

In [ ]:
from rembg import remove

def cutout_pair(closed, opened):
    alpha = remove(closed).split()[-1]
    c, o = closed.convert("RGBA"), opened.convert("RGBA")
    c.putalpha(alpha)
    o.putalpha(alpha)
    return c, o

closed_rgba, open_rgba = cutout_pair(closed, opened)
display(closed_rgba)

## 6. Vista previa animada

In [ ]:
def preview_gif(closed_rgba, open_rgba, path, bg=(225, 233, 245, 255), cycles=8, ms=160):
    base = Image.new("RGBA", closed_rgba.size, bg)
    fc = Image.alpha_composite(base, closed_rgba).convert("RGB")
    fo = Image.alpha_composite(base, open_rgba).convert("RGB")
    frames = [fc, fo] * cycles
    frames[0].save(path, save_all=True, append_images=frames[1:], duration=ms, loop=0)
    return path

gif_path = preview_gif(closed_rgba, open_rgba, f"{AVATAR_DIR}/{PERSONA}_preview.gif")
display(IPImage(filename=gif_path))

## 7. Guardar

In [ ]:
closed_rgba.save(f"{AVATAR_DIR}/{PERSONA}_closed.png")
open_rgba.save(f"{AVATAR_DIR}/{PERSONA}_open.png")

meta = {
    "persona": PERSONA,
    "model": "stabilityai/sdxl-turbo (img2img)",
    "prompt_closed": PROMPT_CLOSED,
    "closed": {k: variants_closed[ELEGIDO_CLOSED][k] for k in ("strength", "seed")},
    "prompt_open": PROMPT_OPEN,
    "open": {k: variants_open[ELEGIDO_OPEN][k] for k in ("strength", "seed")},
    "mouth_box": MOUTH_BOX,
    "plan_b_mouth": USAR_PLAN_B,
    "background_removal": "rembg (U2-Net)",
}
with open(f"{AVATAR_DIR}/{PERSONA}_meta.json", "w") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(sorted(os.listdir(AVATAR_DIR)))

Cuando tengas los avatares de las dos, descarga la carpeta y súbela a `assets/avatar/` en el repo.

In [ ]:
!zip -qr avatars.zip assets/avatar
files.download("avatars.zip")